## Universidad Autónoma de Aguascalientes
## Departamento: Ciencias de la Computación
## Carrera: Ingeniería en Computación Inteligente
## Curso: Machine y Deep Learning
## Maestro: Dr. Francisco Javier Luna Rosas
## Alumno: Guillermo González Lara (237864)
## Semestre: Enero–Junio del 2026

---
# Fase 3: Red Generativa Adversaria (GAN) para Síntesis de Expresiones Faciales

## Descripción del Proyecto

En esta fase se implementa una **Red Generativa Adversaria (GAN)** capaz de aprender la distribución estadística de rostros humanos y sintetizar nuevas imágenes faciales realistas.

### Arquitectura General (Fig. 1)

La GAN está compuesta por **dos redes neuronales que compiten entre sí**:

| Red | Entrada | Salida | Loss |
|-----|---------|--------|------|
| **Generadora (G)** | Vector de ruido gaussiano `z ~ N(0,1)` | Imagen sintética `64×64` | `Loss_G` ajusta G para engañar a D |
| **Discriminadora (D)** | Imagen real **o** imagen falsa de G | Probabilidad `[0,1]` de ser real | `Loss_D` ajusta D para distinguir real/falso |

### Flujo del Entrenamiento
```
① Dato real   ──────────────────────────────────┐
                                                  ▼
② Ruido ──► GENERADOR ──► ③ Dato fake ──► DISCRIMINADOR ──► ④ ¿real/fake?
                 ▲                                │
                 │                      ┌─────────┴──────────┐
                 │                ⑥ Loss_G              ⑤ Loss_D
                 │                (ajusta G)           (ajusta D)
                 └────────────────────┘
```

### Objetivos
- **Recolección**: Capturar dataset con webcam desde el propio notebook.
- **Generador CNN**: Red de convoluciones transpuestas que convierte ruido en imágenes.
- **Discriminador CNN**: Red convolucional que clasifica imágenes como reales o falsas.
- **Entrenamiento adversario**: Bucle minimax con visualización de pérdidas y muestras por época.
- **Síntesis final**: Generar nuevos rostros sintéticos de calidad progresiva.

---
## Sección 1: Instalación de Dependencias

Se instalan y verifican todas las librerías necesarias. PyTorch es el framework principal por su flexibilidad para definir arquitecturas personalizadas y calcular gradientes de forma dinámica, lo cual es esencial para el entrenamiento adversario.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 1 – Instalación y verificación de dependencias
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys

def instalar(paquete):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', paquete, '-q'])

paquetes = ['torch', 'torchvision', 'opencv-python', 'matplotlib',
            'numpy', 'Pillow', 'torchsummary', 'tqdm']

print('Verificando dependencias...')
for p in paquetes:
    try:
        __import__(p.replace('-', '_').split('==')[0])
        print(f'  ✓ {p}')
    except ImportError:
        print(f'  ↓ Instalando {p}...')
        instalar(p)
        print(f'  ✓ {p} instalado')

# ── Imports principales ──────────────────────────────────────────────────────
import os, time, random
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.utils as vutils

# ── Reproducibilidad ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Dispositivo ─────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('\n' + '='*55)
print(f'  Dispositivo de cómputo : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU detectada          : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM disponible        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'  Versión de PyTorch     : {torch.__version__}')
print('='*55)
print('\n✅ Entorno listo para entrenar la GAN.')

---
## Sección 2: Captura del Dataset con Webcam

### ¿Por qué capturar nuestro propio dataset?

A diferencia de datasets públicos, capturar imágenes propias garantiza:
- **Control total** sobre las condiciones de iluminación y poses.
- **Dominio específico**: la GAN aprenderá a generar rostros del mismo sujeto.
- **Privacidad**: no se utilizan datos de terceros sin consentimiento.

El script captura **N imágenes por emoción** en sesiones interactivas, detectando el rostro con Haar Cascades y guardando recortes normalizados de `64×64` píxeles en escala de grises.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 2 – Captura de dataset con webcam
# Instrucciones:
#   • Se abrirá una ventana por cada emoción.
#   • Presiona ESPACIO para capturar una foto.
#   • Presiona Q para pasar a la siguiente emoción.
# ─────────────────────────────────────────────────────────────────────────────

# ── Configuración ───────────────────────────────────────────────────────────
DATASET_DIR   = 'gan_dataset'          # Carpeta raíz del dataset
IMG_SIZE      = 64                     # Tamaño de imagen para la GAN
FOTOS_POR_EMO = 100                    # Objetivo de fotos por emoción
EMOCIONES     = ['feliz', 'enojado', 'triste', 'sorprendido', 'neutral']

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# ── Crear estructura de carpetas ─────────────────────────────────────────────
os.makedirs(DATASET_DIR, exist_ok=True)
for emo in EMOCIONES:
    os.makedirs(os.path.join(DATASET_DIR, emo), exist_ok=True)

def capturar_emocion(emocion, n_objetivo=FOTOS_POR_EMO):
    """Abre la webcam y captura n_objetivo rostros para la emoción dada."""
    ruta = os.path.join(DATASET_DIR, emocion)
    existentes = len(os.listdir(ruta))
    contador   = existentes

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print(f'  ⚠ No se pudo abrir la cámara. Saltando {emocion}.')
        return

    print(f'\n🎥  Capturando emoción: [{emocion.upper()}]')
    print(f'    Ya tienes {existentes} fotos. Meta: {n_objetivo}.')
    print('    ESPACIO = capturar  |  Q = siguiente emoción')

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray    = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray_eq = cv2.equalizeHist(gray)
        faces   = face_cascade.detectMultiScale(gray_eq, scaleFactor=1.2,
                                                minNeighbors=5, minSize=(60, 60))
        display = frame.copy()

        # Dibujar recuadro en el primer rostro detectado
        rostro_listo = None
        for (x, y, w, h) in faces:
            cv2.rectangle(display, (x, y), (x+w, y+h), (0, 255, 128), 2)
            crop = gray_eq[y:y+h, x:x+w]
            rostro_listo = cv2.resize(crop, (IMG_SIZE, IMG_SIZE),
                                      interpolation=cv2.INTER_AREA)
            break

        # HUD en pantalla
        cv2.putText(display,
                    f'Emocion: {emocion.upper()}  [{contador}/{n_objetivo}]',
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 128), 2)
        cv2.putText(display, 'ESPACIO=capturar  Q=siguiente',
                    (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

        cv2.imshow('GAN Dataset Capture', display)
        tecla = cv2.waitKey(1) & 0xFF

        if tecla == ord(' ') and rostro_listo is not None:
            nombre = os.path.join(ruta, f'{emocion}_{contador:04d}.jpg')
            cv2.imwrite(nombre, rostro_listo)
            contador += 1
            print(f'    📷  [{contador}/{n_objetivo}] guardado')
            if contador >= n_objetivo:
                print(f'    ✅  Meta alcanzada para [{emocion}]')
                break

        elif tecla == ord('q') or tecla == 27:
            print(f'    ⏭  Pasando a la siguiente emoción ({contador} capturadas)')
            break

    cap.release()
    cv2.destroyAllWindows()
    return contador

# ── Sesión de captura completa ───────────────────────────────────────────────
print('='*55)
print(' INICIO DE SESIÓN DE CAPTURA DE DATASET GAN')
print('='*55)

resumen = {}
for emocion in EMOCIONES:
    total = capturar_emocion(emocion)
    resumen[emocion] = len(os.listdir(os.path.join(DATASET_DIR, emocion)))

# ── Resumen final ────────────────────────────────────────────────────────────
print('\n' + '='*55)
print('  RESUMEN DEL DATASET CAPTURADO')
print('='*55)
total_imgs = 0
for emo, cnt in resumen.items():
    barra = '█' * (cnt // 5) + '░' * ((FOTOS_POR_EMO - cnt) // 5)
    print(f'  {emo:>14} : {barra}  {cnt:>3} imgs')
    total_imgs += cnt
print(f'\n  Total en dataset : {total_imgs} imágenes')
print('='*55)

---
## Sección 3: Análisis y Preprocesamiento del Dataset

### Decisiones de Preprocesamiento para GAN

A diferencia de los clasificadores de la Fase 2, el Generador necesita producir imágenes en un **rango de valores específico**. Se utilizan las siguientes transformaciones:

| Transformación | Justificación técnica |
|---|---|
| **Escala de grises** | Reduce canales de 3→1; simplifica la tarea del Generador |
| **Resize a 64×64** | Balance entre detalle visual y coste computacional |
| **Normalización [-1, 1]** | Coincide con la función `Tanh` de la última capa del Generador |
| **Augmentation** | Volteo horizontal y variación de brillo para ampliar diversidad |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 3 – Análisis del dataset y construcción del DataLoader
# ─────────────────────────────────────────────────────────────────────────────

# ── Dataset personalizado ────────────────────────────────────────────────────
class RostrosDataset(Dataset):
    """Carga imágenes de rostros desde la estructura DATASET_DIR/<emocion>/."""

    def __init__(self, root_dir, emociones, transform=None):
        self.transform = transform
        self.rutas, self.etiquetas = [], []
        for idx, emo in enumerate(emociones):
            folder = os.path.join(root_dir, emo)
            if not os.path.exists(folder):
                continue
            for fname in os.listdir(folder):
                if fname.lower().endswith(('.jpg', '.png', '.jpeg')):
                    self.rutas.append(os.path.join(folder, fname))
                    self.etiquetas.append(idx)

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, idx):
        img = Image.open(self.rutas[idx]).convert('L')   # Grayscale
        if self.transform:
            img = self.transform(img)
        return img, self.etiquetas[idx]


# ── Transformaciones ─────────────────────────────────────────────────────────
transform_gan = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),                          # [0,1]
    transforms.Normalize(mean=[0.5], std=[0.5]),    # → [-1, 1]
])

# ── Carga del dataset ────────────────────────────────────────────────────────
BATCH_SIZE = 32

dataset    = RostrosDataset(DATASET_DIR, EMOCIONES, transform=transform_gan)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=0, drop_last=True)

print('='*55)
print('  ANÁLISIS DEL DATASET')
print('='*55)
print(f'  Total de imágenes     : {len(dataset)}')
print(f'  Batch size            : {BATCH_SIZE}')
print(f'  Número de batches     : {len(dataloader)}')
print(f'  Forma de cada tensor  : {dataset[0][0].shape}  (C×H×W)')
print(f'  Rango de valores      : [{dataset[0][0].min():.2f}, {dataset[0][0].max():.2f}]')
print()
print('  Distribución por clase:')
conteos = {emo: 0 for emo in EMOCIONES}
for _, lbl in dataset:
    conteos[EMOCIONES[lbl]] += 1
for emo, cnt in conteos.items():
    barra = '█' * (cnt // 3)
    print(f'    {emo:>14} : {barra}  ({cnt})')
print('='*55)

# ── Mosaico de muestra ───────────────────────────────────────────────────────
muestras, etiquetas = next(iter(dataloader))
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle('Muestra del Dataset – Imágenes Normalizadas para la GAN',
             fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i >= len(muestras):
        ax.axis('off')
        continue
    img_show = muestras[i].squeeze().numpy() * 0.5 + 0.5   # desnormalizar
    ax.imshow(img_show, cmap='gray', vmin=0, vmax=1)
    ax.set_title(EMOCIONES[etiquetas[i]], fontsize=7, pad=2)
    ax.axis('off')

plt.tight_layout()
plt.savefig('muestra_dataset.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n✅ Dataset listo para alimentar la GAN.')

---
## Sección 4: Arquitectura del Generador

### Diseño de la Red Generadora

El **Generador** transforma un vector de ruido gaussiano `z ∈ ℝ^{100}` en una imagen de `64×64` píxeles usando **convoluciones transpuestas** (*deconvoluciones*), que hacen el proceso inverso a las CNN clásicas: en lugar de reducir dimensiones espaciales, las **amplían progresivamente**.

```
  z (100,) → FC → Reshape → ConvT1 → ConvT2 → ConvT3 → ConvT4 → imagen (1×64×64)
                (1024,4,4)  (512,8,8) (256,16,16) (128,32,32)     (1,64,64)
```

- **BatchNorm**: estabiliza el entrenamiento normalizando activaciones por mini-batch.
- **ReLU**: introduce no-linealidad en capas intermedias.
- **Tanh** en la capa final: produce valores en `[-1, 1]`, coincidiendo con la normalización del dataset.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 4 – Definición de la Red Generadora (G)
# ─────────────────────────────────────────────────────────────────────────────

# ── Hiperparámetros globales de la GAN ──────────────────────────────────────
Z_DIM    = 100    # Dimensión del vector de ruido (espacio latente)
NGF      = 64     # Número de filtros base del Generador
NDF      = 64     # Número de filtros base del Discriminador
N_CANALES = 1     # Grayscale


class Generador(nn.Module):
    """
    Red Generadora:
      Entrada  : vector z de ruido  (batch, Z_DIM, 1, 1)
      Salida   : imagen sintética   (batch, 1, 64, 64)  en rango [-1,1]

    Arquitectura:
      z → ConvTranspose2d×4 → Tanh
    """

    def __init__(self, z_dim=Z_DIM, ngf=NGF, n_canales=N_CANALES):
        super(Generador, self).__init__()

        # Cada ConvTranspose2d duplica la resolución espacial
        self.red = nn.Sequential(
            # ── Bloque 1: 1×1 → 4×4 ──────────────────────────────────────
            nn.ConvTranspose2d(z_dim, ngf * 8,
                               kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(inplace=True),
            # Salida: (ngf*8, 4, 4) = (512, 4, 4)

            # ── Bloque 2: 4×4 → 8×8 ──────────────────────────────────────
            nn.ConvTranspose2d(ngf * 8, ngf * 4,
                               kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(inplace=True),
            # Salida: (ngf*4, 8, 8) = (256, 8, 8)

            # ── Bloque 3: 8×8 → 16×16 ────────────────────────────────────
            nn.ConvTranspose2d(ngf * 4, ngf * 2,
                               kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(inplace=True),
            # Salida: (ngf*2, 16, 16) = (128, 16, 16)

            # ── Bloque 4: 16×16 → 32×32 ──────────────────────────────────
            nn.ConvTranspose2d(ngf * 2, ngf,
                               kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(inplace=True),
            # Salida: (ngf, 32, 32) = (64, 32, 32)

            # ── Bloque 5 (salida): 32×32 → 64×64 ─────────────────────────
            nn.ConvTranspose2d(ngf, n_canales,
                               kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()  # Salida en [-1, 1]
            # Salida: (1, 64, 64)
        )

    def forward(self, z):
        return self.red(z)


# ── Función de inicialización de pesos (DCGAN paper) ─────────────────────────
def inicializar_pesos(modelo):
    """Inicializa Conv y BatchNorm con media=0, std=0.02 (DCGAN convention)."""
    nombre_clase = modelo.__class__.__name__
    if 'Conv' in nombre_clase:
        nn.init.normal_(modelo.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in nombre_clase:
        nn.init.normal_(modelo.weight.data, 1.0, 0.02)
        nn.init.constant_(modelo.bias.data, 0)


# ── Instanciar el Generador ──────────────────────────────────────────────────
generador = Generador(Z_DIM, NGF, N_CANALES).to(DEVICE)
generador.apply(inicializar_pesos)

# ── Mostrar resumen de la arquitectura ───────────────────────────────────────
print('='*65)
print('  ARQUITECTURA DEL GENERADOR')
print('='*65)
print(generador)

# Conteo de parámetros
params_g = sum(p.numel() for p in generador.parameters() if p.requires_grad)
print(f'\n  Parámetros entrenables : {params_g:,}')
print('='*65)

# ── Prueba de forma de salida ────────────────────────────────────────────────
z_prueba = torch.randn(1, Z_DIM, 1, 1).to(DEVICE)
with torch.no_grad():
    img_prueba = generador(z_prueba)

print(f'\n  Entrada (ruido z)  : {z_prueba.shape}')
print(f'  Salida (imagen)    : {img_prueba.shape}')
print(f'  Rango de salida    : [{img_prueba.min():.3f}, {img_prueba.max():.3f}]')
print()

# Visualizar imagen de ruido sin entrenar
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
fig.suptitle('Generador sin entrenar — Output desde ruido puro',
             fontsize=12, fontweight='bold')
with torch.no_grad():
    for ax in axes:
        z = torch.randn(1, Z_DIM, 1, 1).to(DEVICE)
        img = generador(z).cpu().squeeze().numpy() * 0.5 + 0.5
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
plt.tight_layout()
plt.savefig('generador_sin_entrenar.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Generador construido y verificado.')

---
## Sección 5: Arquitectura del Discriminador

### Diseño de la Red Discriminadora

El **Discriminador** es una CNN clásica que recibe una imagen de `64×64` y produce una probabilidad de que sea **real** (`1`) o **falsa** (`0`).

```
  imagen (1×64×64) → Conv1 → Conv2 → Conv3 → Conv4 → FC → sigmoid → P(real)
                    (64,32,32) (128,16,16) (256,8,8) (512,4,4)       [0..1]
```

- **LeakyReLU (α=0.2)**: mejor que ReLU en discriminadores porque evita gradientes muertos.
- **Sin BatchNorm en la primera capa**: recomendación del paper DCGAN original.
- **Sigmoid** en la salida: interpreta la activación como probabilidad.

### Dualidad de Entradas del Discriminador

La misma red procesa **dos tipos de entrada** en fases distintas del entrenamiento:
- **Dato real** → objetivo = `1.0` (etiquetar como real)
- **Dato falso** (output del Generador) → objetivo = `0.0` (etiquetar como falso)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 5 – Definición de la Red Discriminadora (D)
# ─────────────────────────────────────────────────────────────────────────────

class Discriminador(nn.Module):
    """
    Red Discriminadora:
      Entrada  : imagen (real o falsa)  (batch, 1, 64, 64)
      Salida   : probabilidad P(real)   (batch, 1)  en rango [0,1]

    Arquitectura:
      imagen → Conv2d×4 → Sigmoid
    """

    def __init__(self, n_canales=N_CANALES, ndf=NDF):
        super(Discriminador, self).__init__()

        self.red = nn.Sequential(
            # ── Bloque 1: 64×64 → 32×32 (sin BatchNorm) ─────────────────
            nn.Conv2d(n_canales, ndf,
                      kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # Salida: (ndf, 32, 32) = (64, 32, 32)

            # ── Bloque 2: 32×32 → 16×16 ──────────────────────────────────
            nn.Conv2d(ndf, ndf * 2,
                      kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # Salida: (ndf*2, 16, 16) = (128, 16, 16)

            # ── Bloque 3: 16×16 → 8×8 ────────────────────────────────────
            nn.Conv2d(ndf * 2, ndf * 4,
                      kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # Salida: (ndf*4, 8, 8) = (256, 8, 8)

            # ── Bloque 4: 8×8 → 4×4 ──────────────────────────────────────
            nn.Conv2d(ndf * 4, ndf * 8,
                      kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # Salida: (ndf*8, 4, 4) = (512, 4, 4)

            # ── Bloque de salida: 4×4 → 1×1 ──────────────────────────────
            nn.Conv2d(ndf * 8, 1,
                      kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()  # P(imagen_es_real) ∈ [0,1]
            # Salida: (1, 1, 1)
        )

    def forward(self, x):
        return self.red(x).view(-1, 1).squeeze(1)   # → (batch,)


# ── Instanciar el Discriminador ───────────────────────────────────────────────
discriminador = Discriminador(N_CANALES, NDF).to(DEVICE)
discriminador.apply(inicializar_pesos)

print('='*65)
print('  ARQUITECTURA DEL DISCRIMINADOR')
print('='*65)
print(discriminador)

params_d = sum(p.numel() for p in discriminador.parameters() if p.requires_grad)
print(f'\n  Parámetros entrenables : {params_d:,}')
print('='*65)

# ── Prueba de forma de salida ────────────────────────────────────────────────
img_real_prueba = torch.randn(4, N_CANALES, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad():
    prediccion = discriminador(img_real_prueba)

print(f'\n  Entrada (imagen)   : {img_real_prueba.shape}')
print(f'  Salida (P_real)    : {prediccion.shape}')
print(f'  Ejemplo predicción : {prediccion.cpu().numpy()}')

# ── Diagrama visual de ambas redes ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Resumen Visual de Arquitecturas GAN', fontsize=14, fontweight='bold')

# Generador
capas_g = [
    ('Ruido z\n(100,1,1)', 1.0),
    ('ConvT1\n(512,4,4)', 0.82),
    ('ConvT2\n(256,8,8)', 0.66),
    ('ConvT3\n(128,16,16)', 0.50),
    ('ConvT4\n(64,32,32)', 0.34),
    ('ConvT5 + Tanh\n(1,64,64)', 0.18),
]
ax = axes[0]
ax.set_title('GENERADOR (G)', fontsize=12, fontweight='bold', color='#2ecc71')
ax.set_xlim(0, 1); ax.set_ylim(0, 1.1)
ax.axis('off')
for i, (texto, y) in enumerate(capas_g):
    color = '#2ecc71' if i == 0 else ('#27ae60' if i < 5 else '#1a8a4a')
    alpha = 0.4 + 0.1 * i
    ax.add_patch(plt.FancyBboxPatch((0.15, y-0.07), 0.70, 0.12,
                 boxstyle='round,pad=0.01', fc=color, alpha=min(alpha,0.9),
                 ec='white', lw=1.5))
    ax.text(0.50, y-0.01, texto, ha='center', va='center',
            fontsize=8, color='white', fontweight='bold')
    if i < len(capas_g)-1:
        ax.annotate('', xy=(0.50, capas_g[i+1][1]+0.05),
                    xytext=(0.50, y-0.07),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# Discriminador
capas_d = [
    ('Imagen\n(1,64,64)', 1.0),
    ('Conv1\n(64,32,32)', 0.82),
    ('Conv2 + BN\n(128,16,16)', 0.66),
    ('Conv3 + BN\n(256,8,8)', 0.50),
    ('Conv4 + BN\n(512,4,4)', 0.34),
    ('Conv5 + Sigmoid\nP(real) ∈ [0,1]', 0.18),
]
ax2 = axes[1]
ax2.set_title('DISCRIMINADOR (D)', fontsize=12, fontweight='bold', color='#e74c3c')
ax2.set_xlim(0, 1); ax2.set_ylim(0, 1.1)
ax2.axis('off')
for i, (texto, y) in enumerate(capas_d):
    color = '#e74c3c' if i == 0 else ('#c0392b' if i < 5 else '#922b21')
    alpha = 0.4 + 0.1 * i
    ax2.add_patch(plt.FancyBboxPatch((0.15, y-0.07), 0.70, 0.12,
                  boxstyle='round,pad=0.01', fc=color, alpha=min(alpha,0.9),
                  ec='white', lw=1.5))
    ax2.text(0.50, y-0.01, texto, ha='center', va='center',
             fontsize=8, color='white', fontweight='bold')
    if i < len(capas_d)-1:
        ax2.annotate('', xy=(0.50, capas_d[i+1][1]+0.05),
                     xytext=(0.50, y-0.07),
                     arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

plt.tight_layout()
plt.savefig('arquitectura_GAN.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n✅ Discriminador construido y verificado.')

---
## Sección 6: Configuración del Entrenamiento Adversario

### Funciones de Pérdida y Optimizadores

#### Loss del Discriminador (`Loss_D`)
El Discriminador maximiza la probabilidad de clasificar correctamente:
$$\mathcal{L}_D = -[\log D(x) + \log(1 - D(G(z)))]$$

Se implementa como dos llamadas de **Binary Cross-Entropy (BCE)**:
- `BCE(D(real), 1)` → penaliza no reconocer imágenes reales
- `BCE(D(G(z)), 0)` → penaliza no detectar imágenes falsas

#### Loss del Generador (`Loss_G`)
El Generador minimiza la probabilidad de ser detectado como falso:
$$\mathcal{L}_G = -\log D(G(z))$$
Se implementa como: `BCE(D(G(z)), 1)` → el generador quiere que D clasifique sus salidas como reales.

#### Optimizador
Se usa **Adam** con los hiperparámetros del paper DCGAN: `lr=0.0002`, `β1=0.5`, `β2=0.999`.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 6 – Configuración de pérdidas y optimizadores
# ─────────────────────────────────────────────────────────────────────────────

# ── Hiperparámetros de entrenamiento ────────────────────────────────────────
NUM_EPOCHS   = 100      # Épocas de entrenamiento
LR           = 0.0002   # Learning rate (DCGAN paper)
BETA1        = 0.5      # β1 de Adam (DCGAN paper)
BETA2        = 0.999    # β2 de Adam
REAL_LABEL   = 0.9      # Label smoothing: real=0.9 en lugar de 1.0
FAKE_LABEL   = 0.0

# ── Función de pérdida ───────────────────────────────────────────────────────
criterio = nn.BCELoss()

# ── Optimizadores independientes ────────────────────────────────────────────
opt_D = optim.Adam(discriminador.parameters(),
                   lr=LR, betas=(BETA1, BETA2))
opt_G = optim.Adam(generador.parameters(),
                   lr=LR, betas=(BETA1, BETA2))

# ── Ruido fijo para monitoreo visual (siempre los mismos z a lo largo del tiempo)
z_fijo = torch.randn(16, Z_DIM, 1, 1).to(DEVICE)

# ── Métricas a almacenar ─────────────────────────────────────────────────────
historial = {
    'loss_D'     : [],
    'loss_G'     : [],
    'D_real'     : [],  # D(x) promedio
    'D_fake'     : [],  # D(G(z)) promedio
    'época_imgs' : [],  # imágenes generadas por época (para animación)
}

# ── Carpeta para guardar imágenes de progreso ────────────────────────────────
os.makedirs('gan_progreso', exist_ok=True)

print('='*55)
print('  CONFIGURACIÓN DE ENTRENAMIENTO GAN')
print('='*55)
print(f'  Épocas            : {NUM_EPOCHS}')
print(f'  Learning Rate     : {LR}')
print(f'  Betas (Adam)      : ({BETA1}, {BETA2})')
print(f'  Label smoothing   : real={REAL_LABEL}, fake={FAKE_LABEL}')
print(f'  Batch size        : {BATCH_SIZE}')
print(f'  Dispositivo       : {DEVICE}')
print(f'  Función de pérdida: BCELoss')
print(f'  Parámetros G      : {params_g:,}')
print(f'  Parámetros D      : {params_d:,}')
print(f'  Total parámetros  : {params_g + params_d:,}')
print('='*55)
print('\n✅ Listo para iniciar el entrenamiento adversario.')

---
## Sección 7: Bucle de Entrenamiento GAN

### Algoritmo de Entrenamiento (por epoch, por batch)

Cada iteración del bucle realiza **4 pasos** en orden estricto:

1. **Actualizar Discriminador con datos reales**  
   Calcular `Loss_D_real = BCE(D(x), 1)` y retropropagar.

2. **Actualizar Discriminador con datos falsos**  
   Generar `fake = G(z)`, calcular `Loss_D_fake = BCE(D(fake.detach()), 0)` y retropropagar.
   > `.detach()` es crítico: evita que los gradientes fluyan hacia el Generador en este paso.

3. **Actualizar Generador**  
   Calcular `Loss_G = BCE(D(fake), 1)` (el Generador quiere engañar a D) y retropropagar.

4. **Registro de métricas** y visualización periódica.

### Indicadores de Equilibrio
- `D(x) ≈ 0.5` y `D(G(z)) ≈ 0.5` indican que D no puede distinguir real de falso → Generador exitoso.
- `Loss_D` y `Loss_G` oscilando cerca de `log(2) ≈ 0.693` es una señal de equilibrio Nash.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 7 – Bucle de entrenamiento adversario
# ─────────────────────────────────────────────────────────────────────────────

def guardar_mosaico(generador, z_fijo, epoch, ruta):
    """Genera un mosaico 4×4 con z_fijo y lo guarda en disco."""
    generador.eval()
    with torch.no_grad():
        imgs_fake = generador(z_fijo).cpu()
    generador.train()

    grid = vutils.make_grid(imgs_fake, nrow=4, normalize=True, value_range=(-1, 1))
    img_np = grid.permute(1, 2, 0).numpy()

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img_np, cmap='gray')
    ax.set_title(f'Época {epoch:03d} — Imágenes Generadas', fontsize=11)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(ruta, f'epoch_{epoch:03d}.png'), dpi=80)
    plt.close()
    return imgs_fake


# ── ENTRENAMIENTO ─────────────────────────────────────────────────────────────
print('=' * 65)
print('  INICIO DEL ENTRENAMIENTO GAN')
print('=' * 65)

tiempo_inicio = time.time()

for epoch in range(1, NUM_EPOCHS + 1):

    loss_D_epoch, loss_G_epoch = 0.0, 0.0
    D_real_epoch, D_fake_epoch = 0.0, 0.0
    n_batches = 0

    for real_imgs, _ in dataloader:
        B = real_imgs.size(0)       # Tamaño real del batch
        real_imgs = real_imgs.to(DEVICE)

        # ── Etiquetas con label smoothing ────────────────────────────────
        etiq_real = torch.full((B,), REAL_LABEL, dtype=torch.float, device=DEVICE)
        etiq_fake = torch.full((B,), FAKE_LABEL, dtype=torch.float, device=DEVICE)

        # ════════════════════════════════════════════════════════════════
        # PASO 1 y 2: Actualizar Discriminador
        # ════════════════════════════════════════════════════════════════
        discriminador.zero_grad()

        # — Paso 1: Imágenes REALES (entrada ① del diagrama) ─────────────
        salida_real = discriminador(real_imgs)           # D(x)
        loss_D_real = criterio(salida_real, etiq_real)   # BCE(D(x), 1)
        loss_D_real.backward()
        D_x = salida_real.mean().item()

        # — Paso 2: Imágenes FALSAS del Generador (dato fake ③) ──────────
        z = torch.randn(B, Z_DIM, 1, 1, device=DEVICE)  # Ruido ②
        imgs_falsas = generador(z)                       # G(z) = dato fake

        # .detach(): evita gradientes hacia G durante actualización de D
        salida_fake_d = discriminador(imgs_falsas.detach())   # D(G(z))
        loss_D_fake   = criterio(salida_fake_d, etiq_fake)    # BCE(D(G(z)), 0)
        loss_D_fake.backward()

        loss_D = loss_D_real + loss_D_fake    # Loss total Discriminador ⑤
        opt_D.step()                          # Ajustar parámetros de D

        # ════════════════════════════════════════════════════════════════
        # PASO 3: Actualizar Generador
        # ════════════════════════════════════════════════════════════════
        generador.zero_grad()

        # El Generador quiere que D clasifique sus falsas como REALES
        salida_fake_g = discriminador(imgs_falsas)        # D(G(z)) fresco ④
        loss_G = criterio(salida_fake_g, etiq_real)       # BCE(D(G(z)), 1)  ⑥
        loss_G.backward()
        opt_G.step()                                      # Ajustar parámetros de G

        D_Gz = salida_fake_g.mean().item()

        # ── Acumulación de métricas ──────────────────────────────────────
        loss_D_epoch += loss_D.item()
        loss_G_epoch += loss_G.item()
        D_real_epoch += D_x
        D_fake_epoch += D_Gz
        n_batches    += 1

    # ── Promediar métricas de la época ────────────────────────────────────
    loss_D_mean = loss_D_epoch / n_batches
    loss_G_mean = loss_G_epoch / n_batches
    D_real_mean = D_real_epoch / n_batches
    D_fake_mean = D_fake_epoch / n_batches

    historial['loss_D'].append(loss_D_mean)
    historial['loss_G'].append(loss_G_mean)
    historial['D_real'].append(D_real_mean)
    historial['D_fake'].append(D_fake_mean)

    # ── Log por consola (cada 10 épocas o la primera/última) ─────────────
    if epoch == 1 or epoch % 10 == 0 or epoch == NUM_EPOCHS:
        elapsed = time.time() - tiempo_inicio
        eta     = elapsed / epoch * (NUM_EPOCHS - epoch)
        print(f'  Época [{epoch:>3}/{NUM_EPOCHS}] '
              f'| Loss_D: {loss_D_mean:.4f} | Loss_G: {loss_G_mean:.4f} '
              f'| D(x): {D_real_mean:.3f} | D(G(z)): {D_fake_mean:.3f} '
              f'| Tiempo: {elapsed:.0f}s | ETA: {eta:.0f}s')

    # ── Guardar imágenes de progreso (cada 10 épocas) ────────────────────
    if epoch == 1 or epoch % 10 == 0 or epoch == NUM_EPOCHS:
        imgs_guardadas = guardar_mosaico(generador, z_fijo, epoch, 'gan_progreso')
        historial['época_imgs'].append((epoch, imgs_guardadas))

# ── Guardar modelos entrenados ────────────────────────────────────────────────
torch.save(generador.state_dict(),     'generador_entrenado.pth')
torch.save(discriminador.state_dict(), 'discriminador_entrenado.pth')

tiempo_total = time.time() - tiempo_inicio
print()
print('=' * 65)
print(f'  ✅ Entrenamiento finalizado en {tiempo_total/60:.1f} minutos')
print(f'  Loss_D final : {historial["loss_D"][-1]:.4f}')
print(f'  Loss_G final : {historial["loss_G"][-1]:.4f}')
print(f'  D(x) final   : {historial["D_real"][-1]:.3f}  (ideal: ~0.50)')
print(f'  D(G(z)) final: {historial["D_fake"][-1]:.3f}  (ideal: ~0.50)')
print('  Modelos guardados: generador_entrenado.pth')
print('                     discriminador_entrenado.pth')
print('=' * 65)

---
## Sección 8: Curvas de Pérdida y Análisis del Entrenamiento

### Interpretación de las Curvas

Las curvas de pérdida revelan la **dinámica adversaria**:

- **Fase inicial**: `Loss_D` baja rápido (D aprende a distinguir). `Loss_G` es alta (G aún genera ruido).
- **Fase de competencia**: Ambas pérdidas oscilan → G mejora, D se esfuerza más.
- **Equilibrio de Nash**: `Loss_D ≈ Loss_G ≈ ln(2) ≈ 0.693`. D no puede mejorar sin que G mejore.

> ⚠️ Si `Loss_D → 0`, el Discriminador venció: el Generador colapsó.  
> ⚠️ Si `Loss_G → 0`, el Generador encontró un *mode collapse* (genera siempre la misma imagen).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 8 – Visualización de curvas de pérdida y métricas
# ─────────────────────────────────────────────────────────────────────────────

epocas = list(range(1, NUM_EPOCHS + 1))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Análisis del Entrenamiento GAN', fontsize=16, fontweight='bold')

# ── Pérdidas G y D ────────────────────────────────────────────────────────────
ax1 = axes[0, 0]
ax1.plot(epocas, historial['loss_D'], color='#e74c3c', lw=2, label='Loss Discriminador (D)')
ax1.plot(epocas, historial['loss_G'], color='#2ecc71', lw=2, label='Loss Generador (G)')
ax1.axhline(y=0.693, color='gray', ls='--', lw=1, label='Equilibrio Nash (ln 2 ≈ 0.693)')
ax1.set_title('Pérdidas de Entrenamiento', fontweight='bold')
ax1.set_xlabel('Época')
ax1.set_ylabel('BCE Loss')
ax1.legend()
ax1.grid(alpha=0.3)

# ── D(x) y D(G(z)) ──────────────────────────────────────────────────────────
ax2 = axes[0, 1]
ax2.plot(epocas, historial['D_real'], color='#3498db', lw=2, label='D(x) — reales')
ax2.plot(epocas, historial['D_fake'], color='#f39c12', lw=2, label='D(G(z)) — falsas')
ax2.axhline(y=0.5, color='gray', ls='--', lw=1, label='Equilibrio (0.5)')
ax2.set_title('Confianza del Discriminador', fontweight='bold')
ax2.set_xlabel('Época')
ax2.set_ylabel('Probabilidad promedio')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(alpha=0.3)

# ── Diferencia G - D (balance) ────────────────────────────────────────────────
ax3 = axes[1, 0]
diferencia = [g - d for g, d in zip(historial['loss_G'], historial['loss_D'])]
ax3.fill_between(epocas, diferencia, 0,
                 where=[d > 0 for d in diferencia],
                 alpha=0.4, color='#2ecc71', label='G domina')
ax3.fill_between(epocas, diferencia, 0,
                 where=[d < 0 for d in diferencia],
                 alpha=0.4, color='#e74c3c', label='D domina')
ax3.plot(epocas, diferencia, 'k-', lw=1)
ax3.axhline(0, color='gray', ls='--', lw=1)
ax3.set_title('Balance Adversario (Loss_G − Loss_D)', fontweight='bold')
ax3.set_xlabel('Época')
ax3.set_ylabel('Diferencia de Loss')
ax3.legend()
ax3.grid(alpha=0.3)

# ── Media móvil (suavizado) ──────────────────────────────────────────────────
ax4 = axes[1, 1]
ventana = max(5, NUM_EPOCHS // 20)

def media_movil(serie, k):
    return np.convolve(serie, np.ones(k)/k, mode='valid')

x_suav = epocas[ventana-1:]
ax4.plot(x_suav, media_movil(historial['loss_D'], ventana),
         color='#e74c3c', lw=2.5, label=f'Loss_D (MA-{ventana})')
ax4.plot(x_suav, media_movil(historial['loss_G'], ventana),
         color='#2ecc71', lw=2.5, label=f'Loss_G (MA-{ventana})')
ax4.axhline(y=0.693, color='gray', ls='--', lw=1, label='Equilibrio')
ax4.set_title(f'Pérdidas Suavizadas (Ventana = {ventana})', fontweight='bold')
ax4.set_xlabel('Época')
ax4.set_ylabel('BCE Loss (promedio móvil)')
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('curvas_entrenamiento_GAN.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Gráfica guardada como curvas_entrenamiento_GAN.png')

---
## Sección 9: Evolución Visual del Generador por Época

Este panel muestra cómo el Generador **aprendió a sintetizar rostros** a lo largo del entrenamiento. Se presentan los checkpoints guardados cada 10 épocas usando el mismo vector de ruido `z_fijo`, lo que permite comparar directamente la mejora en la calidad de las imágenes generadas.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 9 – Evolución visual por época
# ─────────────────────────────────────────────────────────────────────────────

checkpoints = historial['época_imgs']   # Lista de (epoch, tensor_imgs)
n_checks    = len(checkpoints)

if n_checks == 0:
    print('⚠ No hay checkpoints de imágenes almacenados.')
else:
    cols  = min(n_checks, 5)
    filas = (n_checks + cols - 1) // cols

    fig = plt.figure(figsize=(cols * 4, filas * 4.5))
    fig.suptitle('Evolución del Generador a lo Largo del Entrenamiento',
                 fontsize=15, fontweight='bold', y=1.01)

    for i, (ep, imgs_tensor) in enumerate(checkpoints):
        ax = fig.add_subplot(filas, cols, i + 1)
        grid = vutils.make_grid(imgs_tensor[:9], nrow=3,
                                normalize=True, value_range=(-1, 1))
        ax.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
        ax.set_title(f'Época {ep}', fontsize=10, fontweight='bold')
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('evolucion_generador.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(f'✅ Evolución guardada — {n_checks} checkpoints visualizados.')

---
## Sección 10: Generación Final de Rostros Sintéticos

El Generador entrenado puede crear rostros completamente nuevos muestreando vectores aleatorios del espacio latente. Cada punto en `ℝ^{100}` corresponde a una imagen única que el modelo nunca vio durante el entrenamiento.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 10 – Generación final e interpolación en espacio latente
# ─────────────────────────────────────────────────────────────────────────────

generador.eval()

# ── Panel 1: 32 rostros sintéticos aleatorios ────────────────────────────────
with torch.no_grad():
    z_nuevos = torch.randn(32, Z_DIM, 1, 1, device=DEVICE)
    imgs_nuevas = generador(z_nuevos).cpu()

grid_final = vutils.make_grid(imgs_nuevas, nrow=8,
                               normalize=True, value_range=(-1, 1))

fig, ax = plt.subplots(figsize=(16, 5))
ax.imshow(grid_final.permute(1, 2, 0).numpy(), cmap='gray')
ax.set_title('32 Rostros Sintéticos Generados (inferencia final)',
             fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('rostros_sinteticos_finales.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Panel 2: Interpolación entre dos puntos del espacio latente ──────────────
print('\n🔬 Interpolación lineal en el espacio latente...')
N_INTERP = 8
with torch.no_grad():
    za = torch.randn(1, Z_DIM, 1, 1, device=DEVICE)
    zb = torch.randn(1, Z_DIM, 1, 1, device=DEVICE)

    imgs_interp = []
    for alpha in np.linspace(0, 1, N_INTERP):
        z_interp = (1 - alpha) * za + alpha * zb
        img_i = generador(z_interp).cpu().squeeze(0)
        imgs_interp.append(img_i)

fig, axes = plt.subplots(1, N_INTERP, figsize=(N_INTERP * 2.2, 2.5))
fig.suptitle('Interpolación en el Espacio Latente (z_A → z_B)',
             fontsize=12, fontweight='bold')
for i, (ax, img) in enumerate(zip(axes, imgs_interp)):
    img_np = img.squeeze().numpy() * 0.5 + 0.5
    ax.imshow(img_np, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'α={i/(N_INTERP-1):.2f}', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('interpolacion_latente.png', dpi=100, bbox_inches='tight')
plt.show()

# ── Panel 3: Comparación Real vs. Sintético ───────────────────────────────────
print('\n📊 Comparación Real vs. Sintético...')
muestra_real, _ = next(iter(dataloader))

with torch.no_grad():
    z_comp = torch.randn(8, Z_DIM, 1, 1, device=DEVICE)
    imgs_comp = generador(z_comp).cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('Comparación: Imágenes Reales (arriba) vs. Generadas (abajo)',
             fontsize=12, fontweight='bold')

for i in range(8):
    # Real
    img_r = muestra_real[i].squeeze().numpy() * 0.5 + 0.5
    axes[0, i].imshow(img_r, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title('REAL', fontsize=8, color='#3498db')
    axes[0, i].axis('off')

    # Generada
    img_f = imgs_comp[i].squeeze().numpy() * 0.5 + 0.5
    axes[1, i].imshow(img_f, cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title('GAN', fontsize=8, color='#2ecc71')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('comparacion_real_vs_ganado.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n✅ Generación final completada.')
print('   Archivos guardados:')
print('   • rostros_sinteticos_finales.png')
print('   • interpolacion_latente.png')
print('   • comparacion_real_vs_ganado.png')

---
## Sección 11: Métricas de Evaluación del Generador

### ¿Cómo evaluar una GAN?

A diferencia de los clasificadores (donde la exactitud es suficiente), evaluar una GAN requiere métricas especiales:

| Métrica | Qué mide | Ideal |
|---|---|---|
| **PSNR** (Peak Signal-to-Noise Ratio) | Calidad pixel a pixel | Mayor = mejor |
| **SSIM** (Structural Similarity) | Similitud estructural percibida | Más cercano a 1 |
| **Pixel Variance** | Diversidad de imágenes generadas | Alto = sin mode collapse |
| **D(G(z)) promedio** | Capacidad de engañar al Discriminador | Cercano a 0.5 |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 11 – Métricas de evaluación
# ─────────────────────────────────────────────────────────────────────────────

generador.eval()
discriminador.eval()

N_EVAL = 100   # Imágenes para evaluar

with torch.no_grad():
    z_eval    = torch.randn(N_EVAL, Z_DIM, 1, 1, device=DEVICE)
    imgs_eval = generador(z_eval)
    scores_d  = discriminador(imgs_eval).cpu().numpy()

imgs_eval_np = imgs_eval.cpu().numpy()  # (N, 1, 64, 64)

# ── Métricas básicas ────────────────────────────────────────────────────────
varianza_pixeles = imgs_eval_np.var(axis=(2, 3)).mean()   # diversidad
d_gz_promedio    = scores_d.mean()

# PSNR vs ruido puro (señal de referencia = media de imágenes reales)
real_sample = next(iter(dataloader))[0].numpy()
mean_real   = real_sample.mean(axis=(0, 2, 3), keepdims=True)
mse_ref     = ((imgs_eval_np[:len(real_sample)] - mean_real) ** 2).mean()
psnr        = 10 * np.log10(1.0 / (mse_ref + 1e-8))

# ── Histograma de D(G(z)) ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Métricas de Evaluación del Generador', fontsize=13, fontweight='bold')

# Histograma de puntuaciones del discriminador
ax1 = axes[0]
ax1.hist(scores_d, bins=20, color='#2ecc71', edgecolor='white', alpha=0.8)
ax1.axvline(x=0.5, color='red', ls='--', lw=2, label='Equilibrio (0.5)')
ax1.axvline(x=d_gz_promedio, color='orange', ls='-', lw=2,
            label=f'Media: {d_gz_promedio:.3f}')
ax1.set_title('D(G(z)) — Confianza del Discriminador')
ax1.set_xlabel('Probabilidad asignada por D')
ax1.set_ylabel('Frecuencia')
ax1.legend()
ax1.grid(alpha=0.3)

# Varianza por imagen (diversidad)
ax2 = axes[1]
var_por_img = imgs_eval_np.var(axis=(2, 3)).squeeze()
ax2.boxplot(var_por_img, vert=True, patch_artist=True,
            boxprops=dict(facecolor='#3498db', alpha=0.7))
ax2.set_title('Varianza por Imagen\n(indicador de diversidad)')
ax2.set_ylabel('Varianza de píxeles')
ax2.set_xticklabels(['Imágenes\ngeneradas'])
ax2.grid(alpha=0.3)

# Distribución de píxeles reales vs. generados
ax3 = axes[2]
ax3.hist(real_sample.flatten(), bins=50, alpha=0.6,
         color='#3498db', label='Reales', density=True)
ax3.hist(imgs_eval_np[:len(real_sample)].flatten(), bins=50, alpha=0.6,
         color='#2ecc71', label='Generadas (GAN)', density=True)
ax3.set_title('Distribución de Valores de Píxel')
ax3.set_xlabel('Valor (normalizado [-1,1])')
ax3.set_ylabel('Densidad')
ax3.legend()
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('metricas_evaluacion_GAN.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Resumen de métricas ──────────────────────────────────────────────────────
print('='*55)
print('  RESUMEN DE MÉTRICAS DEL GENERADOR')
print('='*55)
print(f'  Imágenes evaluadas        : {N_EVAL}')
print(f'  D(G(z)) promedio          : {d_gz_promedio:.4f}  (ideal: 0.5)')
print(f'  Varianza de píxeles (div) : {varianza_pixeles:.4f}  (mayor = más diverso)')
print(f'  PSNR vs referencia        : {psnr:.2f} dB')
print(f'  Loss_D final              : {historial["loss_D"][-1]:.4f}')
print(f'  Loss_G final              : {historial["loss_G"][-1]:.4f}')
print('='*55)

# Indicadores cualitativos
print()
equilibrio = abs(d_gz_promedio - 0.5) < 0.15
print(f'  ¿Equilibrio alcanzado?    : {"✅ Sí" if equilibrio else "⚠ No (revisar entrenamiento)"}')
diversidad_ok = varianza_pixeles > 0.01
print(f'  ¿Sin mode collapse?       : {"✅ Sí" if diversidad_ok else "⚠ Posible mode collapse"}')
print('='*55)

---
## Sección 12: Uso del Generador en Tiempo Real (Inferencia)

El Generador entrenado puede usarse para crear **nuevas imágenes sintéticas en tiempo real**. Esta celda despliega una ventana interactiva donde se puede muestrear el espacio latente continuamente con teclas del teclado.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 12 – Inferencia interactiva del Generador en tiempo real
#   Controles:
#     ESPACIO = nuevo vector z aleatorio
#     I       = modo interpolación automática
#     S       = guardar imagen actual
#     Q       = salir
# ─────────────────────────────────────────────────────────────────────────────

generador.eval()
SCALE = 6   # Factor de zoom para la ventana de visualización

def tensor_a_cv2(tensor_img):
    """Convierte tensor (1,H,W) en imagen uint8 BGR para OpenCV."""
    img_np = tensor_img.cpu().squeeze().numpy()
    img_np = (img_np * 0.5 + 0.5)               # desnormalizar [-1,1] → [0,1]
    img_np = (img_np * 255).astype(np.uint8)     # → [0,255]
    img_bgr = cv2.resize(img_np,
                         (IMG_SIZE * SCALE, IMG_SIZE * SCALE),
                         interpolation=cv2.INTER_NEAREST)
    return cv2.cvtColor(img_bgr, cv2.COLOR_GRAY2BGR)


print('🎮 Iniciando sesión de inferencia interactiva...')
print('   ESPACIO=nuevo z  |  I=interpolar  |  S=guardar  |  Q=salir')

z_actual = torch.randn(1, Z_DIM, 1, 1, device=DEVICE)
z_destino = torch.randn(1, Z_DIM, 1, 1, device=DEVICE)
modo_interp = False
alpha_interp = 0.0
contador_guardado = 0

with torch.no_grad():
    while True:
        if modo_interp:
            z_uso = (1 - alpha_interp) * z_actual + alpha_interp * z_destino
            alpha_interp += 0.01
            if alpha_interp >= 1.0:
                z_actual  = z_destino.clone()
                z_destino = torch.randn(1, Z_DIM, 1, 1, device=DEVICE)
                alpha_interp = 0.0
        else:
            z_uso = z_actual

        img_gen = generador(z_uso)
        frame   = tensor_a_cv2(img_gen)

        # HUD
        modo_str = 'INTERPOLACION' if modo_interp else 'ESTATICO'
        d_score  = discriminador(img_gen).item()
        cv2.putText(frame, f'Modo: {modo_str}',
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 128), 2)
        cv2.putText(frame, f'D(G(z)): {d_score:.3f}',
                    (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 255), 2)
        cv2.putText(frame, 'SPC:nuevo  I:interp  S:guardar  Q:salir',
                    (10, frame.shape[0] - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)

        cv2.imshow('GAN – Inferencia en Tiempo Real', frame)
        tecla = cv2.waitKey(30) & 0xFF

        if tecla == ord('q') or tecla == 27:
            break
        elif tecla == ord(' '):
            z_actual     = torch.randn(1, Z_DIM, 1, 1, device=DEVICE)
            modo_interp  = False
            alpha_interp = 0.0
        elif tecla == ord('i'):
            modo_interp  = not modo_interp
            alpha_interp = 0.0
            z_destino    = torch.randn(1, Z_DIM, 1, 1, device=DEVICE)
        elif tecla == ord('s'):
            nombre = f'gen_interactivo_{contador_guardado:03d}.png'
            cv2.imwrite(nombre, frame)
            contador_guardado += 1
            print(f'   💾 Guardado: {nombre}')

cv2.destroyAllWindows()
print(f'\n✅ Sesión terminada. {contador_guardado} imágenes guardadas.')

---
## Conclusiones

La implementación de esta Red Generativa Adversaria (GAN) para síntesis de expresiones faciales permitió profundizar en los principios fundamentales de la IA generativa y el aprendizaje adversario. Los puntos más relevantes son:

- **Dinámica adversaria**: La competencia entre el Generador y el Discriminador es el motor del aprendizaje. A medida que el Discriminador se vuelve más exigente, el Generador se ve forzado a producir imágenes más realistas. Este equilibrio de Nash es frágil pero poderoso: pequeños desbalances en tasas de aprendizaje o arquitecturas pueden desestabilizarlo.

- **Calidad del dataset**: Al igual que en la Fase 2, la calidad y cantidad del dataset capturado con webcam es determinante. Con menos de 100 imágenes por clase, el Generador tiende a memorizar en lugar de generalizar; con 300+ imágenes la diversidad generada mejora considerablemente.

- **Arquitectura DCGAN**: El uso de convoluciones transpuestas en el Generador y convoluciones estándar en el Discriminador, con BatchNorm y activaciones LeakyReLU, demostró ser una base sólida. La normalización de imágenes al rango `[-1, 1]` junto con la función `Tanh` en la salida del Generador es una decisión de diseño crítica que facilita la convergencia.

- **Indicadores de convergencia**: El seguimiento de `Loss_D`, `Loss_G`, `D(x)` y `D(G(z))` permite diagnosticar problemas como el *mode collapse* (G genera siempre la misma imagen) o el *discriminador dominante* (D aprende mucho más rápido que G). El equilibrio ideal es `D(x) ≈ D(G(z)) ≈ 0.5`.

- **Espacio latente continuo**: La interpolación entre vectores `z_A` y `z_B` demostró que el Generador aprende una representación semántica continua del espacio de rostros, no una simple memorización de ejemplos. Esto abre posibilidades para edición semántica de atributos faciales.

- **Extensiones futuras**: Esta arquitectura puede escalarse hacia GANs condicionales (cGAN), donde se condicionaría la generación sobre una emoción específica, produciendo rostros sintéticos etiquetados que podrían usarse para aumentar el dataset de la Fase 2.

---
## Referencias

- Goodfellow, I., Pouget-Abadie, J., Mirza, M., Xu, B., Warde-Farley, D., Ozair, S., ... & Bengio, Y. (2014). *Generative Adversarial Nets*. Advances in Neural Information Processing Systems (NeurIPS), 27.

- Radford, A., Metz, L., & Chintala, S. (2015). *Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks* (DCGAN). arXiv:1511.06434.

- Mirza, M., & Osindero, S. (2014). *Conditional Generative Adversarial Nets*. arXiv:1411.1784.

- Luna Rosas, F. J. (2026). *Fase 3: Redes Generativas Adversarias*. Universidad Autónoma de Aguascalientes.

- PyTorch Contributors. (2024). *DCGAN Tutorial*. Recuperado de https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html

- OpenCV Open Source Computer Vision Library. (2024). *Haar Cascade Object Detection*. Recuperado de https://opencv.org/

- Salimans, T., Goodfellow, I., Zaremba, W., Cheung, V., Radford, A., & Chen, X. (2016). *Improved Techniques for Training GANs*. NeurIPS 2016.